In [ ]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/pytorch)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#

In [ ]:
from typing import Any

import torch as tr
import torch.optim as optim

In [ ]:
def _equal(a, b):
    """
    Compare two numbers or two tensors.
    """

    if (isinstance(a, tr.Tensor) == True) and (isinstance(b, tr.Tensor) == False):
        b = tr.tensor(b)
    elif (isinstance(a, tr.Tensor) == False) and (isinstance(b, tr.Tensor) == True):
        a = tr.tensor(a)

    if (isinstance(a, tr.Tensor) == True) and (isinstance(b, tr.Tensor) == True):
        result = tr.equal(a, b)

        if isinstance(result, bool):
            return result
        else:
            return tr.all(result)
    else:
        return a == b
    

def _almost(a, b, atol=0.01):
    """
    Compare two numbers or two tensors for approximate equality, with a tolerance of `atol`. 
    """

    if (isinstance(a, tr.Tensor) == True) and (isinstance(b, tr.Tensor) == False):
        b = tr.tensor(b)
    elif (isinstance(a, tr.Tensor) == False) and (isinstance(b, tr.Tensor) == True):
        a = tr.tensor(a)

    if (isinstance(a, tr.Tensor) == True) and (isinstance(b, tr.Tensor) == True):
        return tr.all(tr.abs(a - b) <= atol)
    else:
        return abs(a - b) <= atol


def check_eq(a: Any, b: Any, atol=None):
    if atol is None:
        return _equal(a, b)
    else:
        return _almost(a, b, atol=atol)


def assert_eq(a: Any, b: Any, atol=None):
    if atol is None:
        if _equal(a, b) == False:
            raise AssertionError(f"Assertion {a} == {b} failed")
    else:
        if _almost(a, b, atol=atol) == False:
            raise AssertionError(f"Assertion {a} ~ {b} failed (atol={atol})")


def assert_ne(a: Any, b: Any):
    if (a != b) == False:
        raise AssertionError(f"Assertion {a} != {b} failed")
    

def assert_lt(a: Any, b: Any):
    if (a < b) == False:
        raise AssertionError(f"Assertion {a} < {b} failed")

    
def assert_le(a: Any, b: Any):
    if (a <= b) == False:
        raise AssertionError(f"Assertion {a} <= {b} failed")


def assert_gt(a: Any, b: Any):
    if (a > b) == False:
        raise AssertionError(f"Assertion {a} > {b} failed")


def assert_ge(a: Any, b: Any):
    if (a >= b) == False:
        raise AssertionError(f"Assertion {a} >= {b} failed")
    

In [ ]:
#
# Create a PyTorch tensor from a number or a list of numbers.
#

def T(x):
    """
    Returns a PyTorch tensor created from `x`.
    """

    if isinstance(x, tr.Tensor):
        return x
   
    return tr.tensor(x, dtype=tr.float32)


def test_T():
    assert_eq(T(1/3), 0.33333, atol=0.01)
    assert_eq(T([1/1, 1/2, 1/3]), tr.tensor([1, 0.5, 0.33333]), atol=0.01)

In [ ]:
#
# Return a histogram of a vector if integers as a tuple of (unique_values, counts).
#

def count(vector):
    if isinstance(vector, tr.Tensor) == False:
        vector = tr.tensor(vector, dtype=tr.long)

    return vector.unique(sorted=True, return_counts=True)


def test_count():
    actual = count([1, 1, 2, 1, 2, 3, 1, 2, 3, 4, 1, 2, 3, 4, 5])
    expected = (tr.tensor([1, 2, 3, 4, 5]), tr.tensor([5, 4, 3, 2, 1]))

    assert_eq(actual[0], expected[0])
    assert_eq(actual[1], expected[1])

In [ ]:
class Patient:
    def __init__(self, probs = [1/2, 1/4, 1/4]):
        probs = T(probs)
        probs = probs / probs.sum()

        sick = 1 - probs[0]
        bacteria = probs[1]
        virus = probs[2]

        if sick == 1.0:
            temp = 38 + tr.rand(1) * 2
        else:
            temp = 36 + tr.rand(1) * 2

        cough = tr.rand(1) > (1.21 - sick)
        sore_throat = tr.rand(1) > (1.21 - sick)
        headache = tr.rand(1) > (0.5)
                
        is_sick = (temp / 100) + (cough * 0.1) + (sore_throat * 0.1) + 0.12 > 0.5

        if is_sick == True:
            crp = 50 + 150 * (tr.rand(1) > (virus / (bacteria + virus)))
        else:
            crp = 25 + 25 * tr.rand(1)
           
        if crp >= 100:
            rhinitis = 0
        else:
            rhinitis = 1

        # 0 - healthy, 1 - bacteria, 2 - viral
        diagnosis = int(is_sick) * (2 - int(crp >= 100))


        self.data = [temp, rhinitis, cough.item(), sore_throat.item(), headache.item(), crp.item(), diagnosis]

    def temp(self):
        return self.data[0]
    
    def rhinitis(self):
        return self.data[1]
    
    def cough(self):
        return self.data[2]
    
    def sore_throat(self):
        return self.data[3]
    
    def headache(self):
        return self.data[4]
    
    def crp(self):
        return self.data[5]

    def diagnosis(self):
        return self.data[6]
    
    def label(self):
        return ["healthy", "bacterial", "viral"][int(self.diagnosis())]
    
    def __str__(self):
        return f"Patient(temp={self.temp()}, rhinitis={self.rhinitis()}, cough={self.cough()}, sore_throat={self.sore_throat()}, " \
            f"headache={self.headache()}, crp={self.crp()}, diagnosis={self.diagnosis()})"
    

def test_patient():
    assert_eq(sum(Patient([1.00, 0.00, 0.00]).diagnosis() for _ in range(1000)), (T([1000, 0, 0]) @ T([0, 1, 2])).sum().item(), atol=0)  
    assert_eq(sum(Patient([0.00, 1.00, 0.00]).diagnosis() for _ in range(1000)), (T([0, 1000, 0]) @ T([0, 1, 2])).sum().item(), atol=0)
    assert_eq(sum(Patient([0.00, 0.00, 1.00]).diagnosis() for _ in range(1000)), (T([0, 0, 1000]) @ T([0, 1, 2])).sum().item(), atol=0)
    assert_eq(sum(Patient([0.50, 0.00, 0.50]).diagnosis() for _ in range(1000)), (T([500, 0, 500]) @ T([0, 1, 2])).sum().item(), atol=100)
    assert_eq(sum(Patient([0.50, 0.25, 0.25]).diagnosis() for _ in range(1000)), (T([500, 250, 250]) @ T([0, 1, 2])).sum().item(), atol=75)
    assert_eq(sum(Patient([0.50, 0.50, 0.00]).diagnosis() for _ in range(1000)), (T([500, 500, 0]) @ T([0, 1, 2])).sum().item(), atol=50)

test_patient()

In [ ]:
def average_run(N, fn):
    return sum(fn() for _ in range(N)) / N

In [ ]:
def test_perceptron_xy(perceptron, x, y, epochs=500, lr=0.1):
    model = perceptron
    optimizer = optim.SGD(model.parameters(), lr=lr)

    for _ in range(epochs):
        optimizer.zero_grad()
        loss = model.learn(x, y)
        loss.backward()
        optimizer.step()

    with tr.no_grad():
        return (model.predict(x) == y).mean(dtype=tr.float32).item()


def test_perceptron_boolean(perceptron, bool_fn, samples=100, epochs=500, lr=0.1):
    x = (tr.randn(samples, 2, dtype=tr.float32) > 0).float()
    y = bool_fn(x[:, 0], x[:, 1]).float().unsqueeze(1)
    return test_perceptron_xy(perceptron, x, y, epochs, lr=lr)

In [ ]:
if __name__ == "__main__":
    test_T()
    test_count()
    test_patient()
